In [ ]:
# import sys
# print(sys.executable)

/home/lgwalker/genai/.venv/bin/python


# Notebook 1: Tokenizer Training and Data Prep

##### In this notebook we will create the foundational tokenizer and prepare the pre-training corpus.

- Load Dataset: Use load_dataset to pull 50,000 Java methods from `code_search_net`.
- Filter Data: Keep methods with 10–512 tokens.
- Train Tokenizer: Use the sentencepiece library with `model_type='unigram`' and a `vocab_size=16384`.
- Special Tokens: Add `<pad>`, `</s>`, `<unk>`, and the 100 sentinel tokens (`<extra_id_0>` to `<extra_id_99>`).
- Save: Export the tokenizer files to be reused in all notebooks

Directories:

- `./final_pretrained_model`
- `./java_tokenizer`
- `./t5_pretrain`

In [2]:
!pip install datasets transformers sentencepiece tokenizers accelerate torch faiss-cpu sentence-transformers

  Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl (646 kB)
  Using cached requests-2.33.1-py3-none-any.whl (64 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached multiprocess-0.70.19-py310-none-any.whl (134 kB)
  Using cached typer-0.24.2-py3-none-any.whl (55 kB)
  Using cached aiohttp-3.13.5-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.7 MB)
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)


  Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
  Using cached rich-15.0.0-py3-none-any.whl (310 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)


In [3]:
import os
import random
import sentencepiece as spm
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import Metaspace as MetaspaceDecoder
from tokenizers.models import Unigram
from tokenizers.pre_tokenizers import Metaspace
from transformers import (
    PreTrainedTokenizerFast,
    T5Config,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
)

/home/lgwalker/genai/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
dirs = [
    "./java_tokenizer",         # Saved tokenizer
    "./final_pretrained_model", # Final pre-trained model checkpoint
    "./t5_pretrain",            # Trainer output dir
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"Created: {d}")

print("\nAll directories ready.")

Created: ./java_tokenizer
Created: ./final_pretrained_model
Created: ./t5_pretrain

All directories ready.


In [11]:
# Load Dataset (50,000 Java methods from CodeSearchNet)

print("Loading CodeSearchNet Java dataset...")
csn = load_dataset("code_search_net", "java", split="train").shuffle(seed=42).select(range(50000))
print(f"Loaded {len(csn)} Java methods.")

# Write corpus to file with one function per line, newlines replaced with spaces

corpus_path = "java_methods.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for code in csn["whole_func_string"]:
        f.write(code + "\n")

print(f"Corpus written to {corpus_path}.")

Loading CodeSearchNet Java dataset...
Loaded 50000 Java methods.
Corpus written to java_methods.txt.


In [12]:
# Train SentencePiece Unigram Tokenizer

# Requires 100 sentinel tokens for span corruption: <extra_id_0> ... <extra_id_99>
sentinel_tokens = [f"<extra_id_{i}>" for i in range(100)]

spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix="sp_code",
    vocab_size=16384,           # 16,384 tokens
    model_type="unigram",       # Unigram algorithm
    user_defined_symbols=",".join(sentinel_tokens),  # Required for span corruption
    pad_id=0,                   # <pad> = 0
    eos_id=1,                   # </s>  = 1
    unk_id=2,                   # <unk> = 2
    bos_id=-1,                  # T5 does not use a BOS token
    hard_vocab_limit=False,      # allow slight deviation
    character_coverage=1.0     # Cover all characters in the corpus
)

print("SentencePiece tokenizer trained. Files: sp_code.model, sp_code.vocab")



SentencePiece tokenizer trained. Files: sp_code.model, sp_code.vocab


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: java_methods.txt
  input_format: 
  model_prefix: sp_code
  model_type: UNIGRAM
  vocab_size: 16384
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <extra_id_0>
  user_defined_symbols: <extra_id_1>
  user_defined_symbols: <extra_id_2>
  user_defined_symbols: <extra_id_3>
  user_defined_symbols: <extra_id_4>
  user_defined_symbols: <extra_id_5>
  user_defined_symbols: <extra_id_6>
  user_defined_symbols: <extra_id_7>
  user_defined_symbols: <extra_id_8>
  user_defined_symbols: <e

trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_90>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_91>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_92>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_93>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_94>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_95>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_96>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_97>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_98>
trainer_interface.cc(427) LOG(INFO) Adding meta_piece: <extra_id_99>
trainer_interface.cc(432) LOG(INFO) Normalizing sentences...
trainer_interface.cc(541) LOG(INFO) all chars count=27983998
trainer_interface.cc(562) LOG(INFO) Alphabet size=1187
trainer_interface.cc(563) LOG(INFO) Final character coverage=1
trainer_interface.cc(594) LOG(INFO) Done! preprocessed 809875 sentence

In [13]:
# Convert SentencePiece to HuggingFace Fast Tokenizer and Save
sp = spm.SentencePieceProcessor()
sp.Load("sp_code.model")

# Extract vocab as list of (piece, score) tuples required by HF Unigram
vocab = [(sp.IdToPiece(i), sp.GetScore(i)) for i in range(sp.GetPieceSize())]

# Build HF tokenizer object
tokenizer_obj = Tokenizer(Unigram(vocab, unk_id=2))

# Metaspace handles whitespace the same way SentencePiece does (▁ prefix)
tokenizer_obj.pre_tokenizer = Metaspace()
tokenizer_obj.decoder = MetaspaceDecoder()

sentinel_tokens = [f"<extra_id_{i}>" for i in range(100)]

# Wrap as a PreTrainedTokenizerFast with correct special tokens
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_obj,
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    additional_special_tokens=sentinel_tokens,
)

tokenizer.save_pretrained("./java_tokenizer")
print(f"Tokenizer saved to ./java_tokenizer  |  Vocab size: {len(tokenizer)}")



Tokenizer saved to ./java_tokenizer  |  Vocab size: 16384


In [14]:
# Tokenization Examples check

examples = [
    "public void sort(int[] arr) { Arrays.sort(arr); }",
    "private String getName() { return this.name; }",
]

print("\n--- Tokenizer Check ---")
for ex in examples:
    tokens = tokenizer.tokenize(ex)
    ids = tokenizer.encode(ex)
    print(f"\nInput : {ex}")
    print(f"Tokens: {tokens}")
    print(f"IDs   : {ids}")

# Confirm sentinel tokens are present
sentinel_id = tokenizer.convert_tokens_to_ids("<extra_id_0>")
print(f"\n<extra_id_0> token ID: {sentinel_id}  (must NOT be <unk> ID=2)")
assert sentinel_id != 2, "ERROR: Sentinel tokens not found in vocab!"
print("Sentinel token check passed.")


--- Tokenizer Check ---

Input : public void sort(int[] arr) { Arrays.sort(arr); }
Tokens: ['▁public', '▁void', '▁sort', '(', 'int', '[]', '▁arr', ')', '▁{', '▁Arrays', '.', 'sort', '(', 'arr', ');', '▁}']
IDs   : [124, 146, 1842, 104, 182, 165, 2116, 109, 108, 1416, 103, 1501, 104, 3033, 111, 106]

Input : private String getName() { return this.name; }
Tokens: ['▁private', '▁String', '▁', 'getName', '()', '▁{', '▁return', '▁this', '.', 'name', ';', '▁}']
IDs   : [208, 129, 105, 282, 130, 108, 117, 141, 103, 233, 114, 106]

<extra_id_0> token ID: 3  (must NOT be <unk> ID=2)
Sentinel token check passed.


In [16]:
# Vocab Inspection
print("--- Top 20 Tokens in Vocabulary ---")
# Get the first 20 items in the vocab
vocab_list = list(tokenizer.get_vocab().items())
sorted_vocab = sorted(vocab_list, key=lambda x: x[1])

for token, token_id in sorted_vocab[:20]:
    # We use repr() to see hidden characters like the metaspace
    print(f"ID {token_id:5} : {repr(token)}")


--- Top 20 Tokens in Vocabulary ---
ID     0 : '<pad>'
ID     1 : '</s>'
ID     2 : '<unk>'
ID     3 : '<extra_id_0>'
ID     4 : '<extra_id_1>'
ID     5 : '<extra_id_2>'
ID     6 : '<extra_id_3>'
ID     7 : '<extra_id_4>'
ID     8 : '<extra_id_5>'
ID     9 : '<extra_id_6>'
ID    10 : '<extra_id_7>'
ID    11 : '<extra_id_8>'
ID    12 : '<extra_id_9>'
ID    13 : '<extra_id_10>'
ID    14 : '<extra_id_11>'
ID    15 : '<extra_id_12>'
ID    16 : '<extra_id_13>'
ID    17 : '<extra_id_14>'
ID    18 : '<extra_id_15>'
ID    19 : '<extra_id_16>'


In [17]:
# Define T5Config and Initialize Model from Scratch
# pad_id=0, eos_id=1 match the SentencePiece IDs above.
t5_config = T5Config(
    decoder_start_token_id=0,
    pad_token_id=0,
    eos_token_id=1,
    vocab_size=len(tokenizer),  # Match tokenizer vocab size
    d_model=512,                # T5-small embedding dimension
    d_ff=2048,                  # Feed-forward layer dimension
    d_kv=64,                    # Key/value projection dimension
    num_heads=8,                # Attention heads
    num_layers=6,               # Encoder layers
    num_decoder_layers=6,       # Decoder layers
    feed_forward_proj="gated-gelu",
)


model = T5ForConditionalGeneration(config=t5_config)
# Resize embedding matrix to exactly match tokenizer vocab
model.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel initialized from scratch.")
print(f"Embedding shape : {model.shared.weight.shape}")
print(f"Total parameters: {total_params:,}  (~60M expected for T5-small)")



Model initialized from scratch.
Embedding shape : torch.Size([16384, 512])
Total parameters: 65,028,608  (~60M expected for T5-small)


In [24]:
# Tokenize and Filter Dataset (10–512 tokens)

print("\nTokenizing dataset...")
tokenized_csn = csn.map(
    lambda x: tokenizer(x["whole_func_string"], truncation=True, max_length=512),
    batched=True,
    remove_columns=csn.column_names,
)

before = len(tokenized_csn)
# Keeping only methods with 10–512 tokens
tokenized_csn = tokenized_csn.filter(
    lambda x: len(x["input_ids"]) >= 10 and len(x["input_ids"]) <= 512
)
after = len(tokenized_csn)
print(f"Filtered: {before} to {after} methods (kept {after/before*100:.1f}%)")





Tokenizing dataset...
Filtered: 50000 to 50000 methods (kept 100.0%)


In [25]:
# Dataset Content Check
sample_ids = tokenized_csn[0]["input_ids"]
sample_text = tokenizer.decode(sample_ids)

print("--- Dataset Sample Check ---")
print(f"Decoded Sample:\n{sample_text[:150]}...")

# Validation - If this is False, your model will learn to output squished code
has_whitespace = " " in sample_text or "\n" in sample_text
assert has_whitespace, "ERROR: Tokenized data has no whitespace!"

print("Dataset Sample Check Passed")

--- Dataset Sample Check ---
Decoded Sample:
public void put(String entityTypeId, Entity entity) {<unk>    CombinedEntityCache entityCache = caches.get();<unk>    if (entityCache != null) {<unk> ...
Dataset Sample Check Passed


In [ ]:
# Span Corruption Data Collator

# Implements T5's pre-training objective:
#   Randomly select 15% of tokens
#   Group consecutive selected tokens into spans
#   Replace each span in the input with a unique sentinel token
#   Target = sentinel + original span tokens, ended with </s>
class ManualSpanCorruptionCollator:
    def __init__(self, tokenizer, corruption_rate=0.15):
        self.tokenizer = tokenizer
        self.corruption_rate = corruption_rate
        # Pre-fetch all 100 sentinel token IDs
        self.sentinel_ids = tokenizer.convert_tokens_to_ids(
            [f"<extra_id_{i}>" for i in range(100)]
        )

    def __call__(self, batch):
        input_ids_batch = []
        labels_batch = []

        for item in batch:
            token_ids = item["input_ids"]
            n_tokens = len(token_ids)
            n_mask = max(1, int(n_tokens * self.corruption_rate))

            # Randomly select positions to mask
            mask_indices = sorted(random.sample(range(n_tokens), n_mask))

            # Group consecutive indices into contiguous spans
            spans = []
            if mask_indices:
                current_span = [mask_indices[0]]
                for i in range(1, len(mask_indices)):
                    if mask_indices[i] == mask_indices[i - 1] + 1:
                        current_span.append(mask_indices[i])
                    else:
                        spans.append(current_span)
                        current_span = [mask_indices[i]]
                spans.append(current_span)

            # Map each masked position to its span index
            mask_set = set(mask_indices)
            span_map = {
                idx: s_idx
                for s_idx, span in enumerate(spans)
                for idx in span
            }

            # Build corrupted input replace each span with its sentinel token
            input_ids = []
            prev_span_idx = -1
            for pos, tid in enumerate(token_ids):
                if pos in mask_set:
                    s_idx = span_map[pos]
                    if s_idx != prev_span_idx:
                        # First token of this span insert sentinel
                        input_ids.append(self.sentinel_ids[s_idx])
                        prev_span_idx = s_idx
                    # Remaining tokens in span are dropped from input
                else:
                    input_ids.append(tid)
                    prev_span_idx = -1

            # Build target sentinel + original span tokens then </s>
            labels = []
            for s_idx, span in enumerate(spans):
                labels.append(self.sentinel_ids[s_idx])
                for pos in span:
                    labels.append(token_ids[pos])
            labels.append(self.tokenizer.eos_token_id)

            input_ids_batch.append(torch.tensor(input_ids))
            labels_batch.append(torch.tensor(labels))

        # Pad to longest sequence in batch
        input_ids_padded = torch.nn.utils.rnn.pad_sequence(
            input_ids_batch, batch_first=True, padding_value=0
        )
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels_batch, batch_first=True, padding_value=-100  # -100 = ignore in loss
        )

        return {
            "input_ids": input_ids_padded,
            "labels": labels_padded,
            "attention_mask": (input_ids_padded != 0).long(),
        }


collator = ManualSpanCorruptionCollator(tokenizer, corruption_rate=0.15)
print("Span corruption collator initialized  |  Corruption rate: 15%")


Span corruption collator initialized  |  Corruption rate: 15%


In [ ]:
# Pre-training with 3 Epochs, No Early Stopping, No Checkpoint Selection
# Run for 3 epochs on the full corpus. No validation split,
# no early stopping, no checkpoint selection
# use the final model. Log the training loss per epoch
training_args = TrainingArguments(
    output_dir="./t5_pretrain",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=1e-4,
    # Save only at the end
    save_strategy="no",
    # Log loss every 100 steps to see decrease
    logging_steps=500,
    logging_strategy="epoch",
    # Speed optimizations
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    warmup_steps=500,
    weight_decay=0.01,
    # no use of evaluation during pre-training
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_csn,
    data_collator=collator,
    )


print("\nStarting pre-training...")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Training samples: {len(tokenized_csn)}")
print(f"Epochs: 3  |  Batch size: 16  |  No early stopping\n")

trainer.train()


Starting pre-training...
Device: GPU
Training samples: 50000
Epochs: 3  |  Batch size: 16  |  No early stopping



Step,Training Loss
3125,2.523885
6250,2.453815
9375,2.420010


TrainOutput(global_step=9375, training_loss=2.4659032291666665, metrics={'train_runtime': 1527.7902, 'train_samples_per_second': 98.181, 'train_steps_per_second': 6.136, 'total_flos': 2.4944046827372544e+16, 'train_loss': 2.4659032291666665, 'epoch': 3.0})

In [29]:
# Save Final Pre-trained Model

# Use the final model
# Both model weights and tokenizer are saved together so fine-tuning loads from one

model.save_pretrained("./final_pretrained_model")
tokenizer.save_pretrained("./final_pretrained_model")

print("\nPre-trained model saved to ./final_pretrained_model")
print("Checkpoint for tokenizer and pretraining ")

# Log final training loss for the report
log_history = trainer.state.log_history
train_losses = [(e["epoch"], e["loss"]) for e in log_history if "loss" in e]
print("\nTraining loss log (step, loss):")
for step_info in train_losses[-10:]:  # Print last 10 entries
    print(f"  Epoch {step_info[0]:.2f}  |  Loss: {step_info[1]:.4f}")

Writing model shards: 100%|████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.06s/it]


Pre-trained model saved to ./final_pretrained_model
Checkpoint for tokenizer and pretraining 

Training loss log (step, loss):
  Epoch 1.00  |  Loss: 2.5239
  Epoch 2.00  |  Loss: 2.4538
  Epoch 3.00  |  Loss: 2.4200


## Outputs of all files:

Java methods file:
- `./java_methods.txt`

Sentence Piece:
- `./sp_code.vocab`
- `./sp_code.model`

Java tokenizer:
- `./java_tokenizer/tokenizer_config.json`
- `./java_tokenizer/tokenize.json`

Pre train and Final :
- `./final_pretrained_model/config.json`
- `./final_pretrained_model/generations.json`
- `./final_pretrained_model/model.safetensors`
- `./final_pretrained_model/tokenizer_config.json`
- `./final_pretrained_model/tokenizer.json`
- `./t5_pretrain/checkpoint-2813/`

